In [ ]:
import re
import unicodedata

import pandas as pd
import plotly.graph_objects as go

In [ ]:
CSV_MORTALIDADE = "/home/anna/tcc/Tratamento/Mortalidade/tabela_mortalidade_faixa_etaria.csv"

CSV_CASOS_ANO_DIAGNOSTICO = "/home/anna/tcc/Tratamento/Casos/incidencia _ano_diagnostico.csv"
CSV_CASOS_ESCOLARIDADE    = "/home/anna/tcc/Tratamento/Casos/incidencia _escolaridade.csv"
CSV_CASOS_RACA_COR        = "/home/anna/tcc/Tratamento/Casos/incidencia _raca_cor.csv"
CSV_CASOS_FAIXA_ETARIA    = "/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv"


# - Use None para somar TODOS os municípios
MUNICIPIO        = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE   = None

ANOS_RANGE_MORTALIDADE = (1979, 2024)
ANOS_PASSO_MORTALIDADE = 1

ANOS_RANGE_CASOS = (2013, 2023)
ANOS_PASSO_CASOS = 1

# Mortalidade (barras por faixa etária)
FAIXAS_ETARIAS_MORTALIDADE = [
    "0-9", "10-19", "20-29", "30-39",
    "40-49", "50-59", "60-69", "70 ou mais",
]

# Casos (top N categorias para legibilidade)
TOP_N_CATEGORIAS = 25  # None para mostrar tudo (exceto Ano Diagnóstico, que não corta)

# Cores e estilo (padrão do gráfico de mortalidade)
COR_BARRAS = "#2D9CDB"
COR_PONTOS = "#D7263D"

# Título base
TITULO_MORTALIDADE = "Mortalidade por faixa etária 1979 - 2024"

In [ ]:
# 3) Ajustes / tratamento dos dados (leitura, filtro, somatórios)
# - Calcula:

def _norm_txt(txt: str) -> str:
    return (
        unicodedata.normalize("NFKD", str(txt))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
    )


def _norm_col(col: str) -> str:
    return re.sub(r"\s+", "_", _norm_txt(col))


def _find_first_col(df: pd.DataFrame, candidates_norm: set[str]) -> str | None:
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = [c for c in df.columns if _norm_col(c).startswith("unnamed")]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df


def _fmt_int(n):
    try:
        return f"{int(n):,}".replace(",", ".")
    except Exception:
        return str(n)


def _sum_row_wise(frame: pd.DataFrame, cols: list[str]) -> pd.Series:
    """Replica df[col1] + df[col2] + ... (por linha)."""
    if not cols:
        return pd.Series([0] * len(frame), index=frame.index, dtype="int64")
    return (
        frame[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
        .round()
        .astype(int)
    )


def _col_ano_faixa(ano: int, faixa: str) -> str:
    return f"{int(ano)}_{faixa}"


def _col_total_ano(ano: int) -> str:
    return f"Total_{int(ano)}"


def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica MUNICIPIO / CODIGO_MUNICIPIO / REGIONAL_SAUDE para qualquer CSV (colunas variam)."""
    df = _drop_index_cols(df)
    out = df.copy()

    col_mun = _find_first_col(out, {"municipios", "municipio", "cidades", "cidade"})
    col_cod = _find_first_col(out, {"codigo", "cod"})
    col_reg = _find_first_col(out, {"regional_de_saude", "regional_saude", "regional"})

    if MUNICIPIO is not None:
        if not col_mun:
            raise KeyError("Não encontrei coluna de município para filtrar.")
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]
    if CODIGO_MUNICIPIO is not None:
        if not col_cod:
            raise KeyError("Não encontrei coluna de código para filtrar.")
        out = out[pd.to_numeric(out[col_cod], errors="coerce").astype("Int64") == int(CODIGO_MUNICIPIO)]
    if REGIONAL_SAUDE is not None:
        if not col_reg:
            raise KeyError("Não encontrei coluna de regional para filtrar.")
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]

    return out


def _categoria_sort_key(cat):
    try:
        return (0, int(cat))
    except (ValueError, TypeError):
        return (1, str(cat))


def _year_cols(df: pd.DataFrame) -> list[str]:
    """Encontra colunas que parecem ser anos (só números 4 dígitos)."""
    year_cols = []
    for col in df.columns:
        try:
            y = int(col)
            if 1900 <= y <= 2100:
                year_cols.append(col)
        except (ValueError, TypeError):
            pass
    return sorted(year_cols)


def preparar_ano_diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    year_cols = _year_cols(df)
    if not year_cols:
        raise ValueError("Não encontrei colunas de ano (YYYY) no CSV de ano diagnóstico.")
    serie = df[year_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=0)
    serie = serie[serie > 0]
    out = pd.DataFrame({"ano": serie.index.astype(int), "categoria": serie.index.astype(str), "total": serie.values})
    return out.sort_values("ano").reset_index(drop=True)


def preparar_long_wide_c(df: pd.DataFrame) -> pd.DataFrame:
    long_data = []
    for col in df.columns:
        if "_" not in str(col):
            continue
        parts = str(col).rsplit("_", 1)
        if len(parts) != 2:
            continue
        ano_str, categoria = parts
        try:
            ano = int(ano_str)
        except (ValueError, TypeError):
            continue

        total = pd.to_numeric(df[col], errors="coerce").fillna(0).sum()
        if total > 0:
            long_data.append({"ano": ano, "categoria": categoria, "total": int(total)})

    out = pd.DataFrame(long_data)
    return out.sort_values(["ano", "categoria"]).reset_index(drop=True)


def build_por_categoria_por_ano(df_long: pd.DataFrame, *, anos_range: tuple[int, int] | None, top_n: int | None):
    df_long = df_long.copy()
    df_long["ano"] = pd.to_numeric(df_long["ano"], errors="coerce").astype("Int64")
    df_long["total"] = pd.to_numeric(df_long["total"], errors="coerce")
    df_long = df_long.dropna(subset=["ano", "categoria", "total"])
    df_long["ano"] = df_long["ano"].astype(int)
    df_long = df_long[df_long["total"] > 0]

    if anos_range is not None:
        ano_ini, ano_fim = map(int, anos_range)
        df_long = df_long[(df_long["ano"] >= ano_ini) & (df_long["ano"] <= ano_fim)]
        if df_long.empty:
            raise ValueError("Sem dados dentro do intervalo de anos escolhido.")

    por_categoria = df_long.groupby("categoria", as_index=True)["total"].sum()
    por_categoria = por_categoria[por_categoria > 0]
    por_categoria = por_categoria.sort_index(key=lambda idx: pd.Index(sorted(idx, key=_categoria_sort_key)))

    if top_n is not None:
        por_categoria = por_categoria.sort_values(ascending=False).head(int(top_n)).sort_values()

    por_ano = df_long.groupby("ano", as_index=True)["total"].sum().sort_index()
    por_ano = por_ano[por_ano > 0]

    return por_categoria, por_ano


print("=== CONFIG / FILTRO ===")
print("MUNICIPIO:", MUNICIPIO)
print("CODIGO_MUNICIPIO:", CODIGO_MUNICIPIO)
print("REGIONAL_SAUDE:", REGIONAL_SAUDE)
print("ANOS_RANGE_MORTALIDADE:", ANOS_RANGE_MORTALIDADE)
print("ANOS_RANGE_CASOS:", ANOS_RANGE_CASOS)

# Mortalidade (CSV wide)

df_mort = df_mortalidade.copy() if "df_mortalidade" in globals() else pd.read_csv(CSV_MORTALIDADE)
f_mort = aplicar_filtros(df_mort)

print(f"Linhas selecionadas (mortalidade): {len(f_mort)}")

ano_ini_m, ano_fim_m = map(int, ANOS_RANGE_MORTALIDADE)
anos_m = list(range(ano_ini_m, ano_fim_m + 1))

# Barras: soma por faixa etária (replica soma manual por ano e depois soma)
df_faixas_por_linha_mort = pd.DataFrame(index=f_mort.index)
for fx in FAIXAS_ETARIAS_MORTALIDADE:
    cols_fx = [_col_ano_faixa(a, fx) for a in anos_m if _col_ano_faixa(a, fx) in f_mort.columns]
    df_faixas_por_linha_mort[fx] = _sum_row_wise(f_mort, cols_fx)
por_faixa_mort = df_faixas_por_linha_mort[FAIXAS_ETARIAS_MORTALIDADE].sum(axis=0).astype(int)

# Pontos: total por ano (prefere Total_YYYY, se existir)
por_ano_mort = {}
for ano in anos_m:
    col_total = _col_total_ano(ano)
    if col_total in f_mort.columns:
        por_ano_mort[ano] = int(pd.to_numeric(f_mort[col_total], errors="coerce").fillna(0).sum())
    else:
        cols = [_col_ano_faixa(ano, fx) for fx in FAIXAS_ETARIAS_MORTALIDADE if _col_ano_faixa(ano, fx) in f_mort.columns]
        por_ano_mort[ano] = int(_sum_row_wise(f_mort, cols).sum())
por_ano_mort_s = pd.Series(por_ano_mort).sort_index()

print("Mortalidade — exemplo 40-49:", _fmt_int(int(por_faixa_mort.get("40-49", 0))))
if 2023 in por_ano_mort_s.index:
    print("Mortalidade — Total 2023:", _fmt_int(int(por_ano_mort_s.loc[2023])))

# Casos

graficos_casos = []

def _carregar_e_preparar_casos(caminho_csv: str, nome: str, *, modo: str, top_n: int | None, anos_range: tuple[int, int] | None = ANOS_RANGE_CASOS):
    df_raw = pd.read_csv(caminho_csv)
    df_f = aplicar_filtros(df_raw)
    print(f"Linhas selecionadas ({nome}): {len(df_f)}")
    if modo == "ano_cols":
        df_long = preparar_ano_diagnostico(df_f)
    elif modo == "wide_c":
        df_long = preparar_long_wide_c(df_f)
    else:
        raise ValueError("modo inválido")
    por_cat, por_ano = build_por_categoria_por_ano(df_long, anos_range=anos_range, top_n=top_n)
    return por_cat, por_ano

# Ano diagnóstico (colunas YYYY):

por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_ANO_DIAGNOSTICO,
    "Casos - Ano Diagnóstico",
    modo="ano_cols",
    top_n=None,
    anos_range=None,
 )
por_ano_ref = por_ano[(por_ano.index >= ANOS_RANGE_CASOS[0]) & (por_ano.index <= ANOS_RANGE_CASOS[1])]
if por_ano_ref.empty:
    raise ValueError("Sem dados de Ano Diagnóstico no intervalo de referência de casos (2013–2023).")
graficos_casos.append({
    "titulo": "Casos - Ano Diagnóstico",
    "ylabel_categoria": "Ano do diagnóstico",
    "por_categoria": por_cat,
    "por_ano": por_ano_ref,
    "anos_range": ANOS_RANGE_CASOS,
    "anos_passo": ANOS_PASSO_CASOS,
})

# Escolaridade 
por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_ESCOLARIDADE,
    "Casos - Escolaridade",
    modo="wide_c",
    top_n=TOP_N_CATEGORIAS,
 )
graficos_casos.append({
    "titulo": "Casos - Escolaridade",
    "ylabel_categoria": "Escolaridade",
    "por_categoria": por_cat,
    "por_ano": por_ano,
    "anos_range": ANOS_RANGE_CASOS,
    "anos_passo": ANOS_PASSO_CASOS,
})

# Raça/Cor 
por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_RACA_COR,
    "Casos - Raça/Cor",
    modo="wide_c",
    top_n=TOP_N_CATEGORIAS,
 )
graficos_casos.append({
    "titulo": "Casos - Raça/Cor",
    "ylabel_categoria": "Raça/Cor",
    "por_categoria": por_cat,
    "por_ano": por_ano,
    "anos_range": ANOS_RANGE_CASOS,
    "anos_passo": ANOS_PASSO_CASOS,
})

# Faixa Etária 
por_cat, por_ano = _carregar_e_preparar_casos(
    CSV_CASOS_FAIXA_ETARIA,
    "Casos - Faixa Etária",
    modo="wide_c",
    top_n=TOP_N_CATEGORIAS,
 )
graficos_casos.append({
    "titulo": "Casos - Faixa Etária",
    "ylabel_categoria": "Faixa etária",
    "por_categoria": por_cat,
    "por_ano": por_ano,
    "anos_range": ANOS_RANGE_CASOS,
    "anos_passo": ANOS_PASSO_CASOS,
})

print(f"\n Casos: {len(graficos_casos)} gráficos preparados")

In [ ]:
# 4) Design e layout dos gráficos (Plotly)

def plot_ano_diagnostico(
    por_ano: pd.Series,
    titulo: str,
    width: int = 1400,
    height: int = 620,
    cor_barras: str = COR_BARRAS,
):
    por_ano = pd.to_numeric(por_ano, errors="coerce").fillna(0)
    por_ano = por_ano[por_ano > 0]

    anos = por_ano.index.astype(int).tolist()
    valores = por_ano.values.astype(int).tolist()

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=valores,
        y=[str(a) for a in anos],
        orientation="h",
        marker=dict(
            color=cor_barras,
            opacity=0.85,
            line=dict(color="white", width=0.8),
        ),
        text=[_fmt_int(v) for v in valores],
        textposition="outside",
        hovertemplate="<b>Ano:</b> %{y}<br><b>Total:</b> %{x}<extra></extra>",
    ))

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            pad=dict(b=80),
        ),
        height=height,
        width=width,
        plot_bgcolor="#f8f9fb",
        paper_bgcolor="white",
        margin=dict(l=110, r=60, t=80, b=80),
        xaxis=dict(
            title="Número de casos",
            gridcolor="#dde3ed",
            zeroline=True,
        ),
        yaxis=dict(
            title="Ano do diagnóstico",
            type="category",
            categoryorder="array",
            categoryarray=[str(a) for a in anos],
        ),
    )

    fig.show()

def _make_xticks(x_max_data: float) -> list[int]:
    x_max_data = float(x_max_data)
    if x_max_data <= 0:
        return [0, 1]
    step = 500 if x_max_data > 5000 else (200 if x_max_data > 2000 else 100)
    xticks_vals = list(range(1, 11)) + list(range(step, int(x_max_data * 1.15) + step, step))
    xticks_vals = sorted(set(int(v) for v in xticks_vals if v > 0 and v <= int(x_max_data * 1.15)))
    return xticks_vals or [1]


def plot_duplo_plotly(
    *,
    por_categoria: pd.Series,
    por_ano: pd.Series,
    titulo: str,
    xaxis_title: str,
    yaxis_title: str,
    yaxis2_title: str = "Ano",
    anos_range: tuple[int, int],
    anos_passo: int = 1,
    usar_y2: bool = True,
    cor_barras: str = COR_BARRAS,
    cor_pontos: str = COR_PONTOS,
    width: int = 1400,
    height: int = 620,
    force_y_ticklabels: bool = True,
 ):
    por_categoria = por_categoria.copy()
    por_ano = por_ano.copy()

    # Sanitiza
    por_categoria = pd.to_numeric(por_categoria, errors="coerce").fillna(0)
    por_ano = pd.to_numeric(por_ano, errors="coerce").fillna(0)
    por_categoria = por_categoria[por_categoria > 0]
    por_ano = por_ano[por_ano > 0]
    if por_categoria.empty or por_ano.empty:
        raise ValueError("Sem dados > 0 para plotar (categoria/ano).")

    categorias = por_categoria.index.astype(str).tolist()
    valores_barras = por_categoria.values.astype(int).tolist()

    anos_lista = por_ano.index.astype(int).tolist()
    valores_ano = por_ano.values.astype(int).tolist()

    # Tooltips
    ano_ini, ano_fim = map(int, anos_range)
    tooltips_barras = [
        f"<b>{yaxis_title}:</b> {cat}<br><b>Total:</b> {_fmt_int(val)}<br><i>Soma {ano_ini}–{ano_fim}</i>"
        for cat, val in zip(categorias, valores_barras)
    ]
    tooltips_pontos = [
        f"<b>Ano:</b> {ano}<br><b>Total:</b> {_fmt_int(val)}"
        for ano, val in zip(anos_lista, valores_ano)
    ]

    x_max_data = max(max(valores_barras), max(valores_ano))
    xticks_vals = _make_xticks(x_max_data)

    fig = go.Figure()

    # Barras (Y esquerdo)
    fig.add_trace(go.Bar(
        name=f"Total por {yaxis_title} ({ano_ini}–{ano_fim})",
        x=valores_barras,
        y=categorias,
        orientation="h",
        marker=dict(
            color=cor_barras,
            opacity=0.85,
            line=dict(color="white", width=0.8),
        ),
        text=[_fmt_int(v) for v in valores_barras],
        textposition="outside",
        textfont=dict(size=11, color="#222"),
        hovertext=tooltips_barras,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis="y",
        xaxis="x",
    ))

    # Pontos
    if usar_y2:
        y_scatter = anos_lista
        yaxis_scatter = "y2"
        name_scatter = f"Total por ano ({ano_ini}–{ano_fim})"
    else:
        # Para Ano Diagnóstico: usa o mesmo Y categórico das barras (anos como categorias)
        y_scatter = [str(a) for a in anos_lista]
        yaxis_scatter = "y"
        name_scatter = f"Total por ano ({ano_ini}–{ano_fim})"

    fig.add_trace(go.Scatter(
        name=name_scatter,
        x=valores_ano,
        y=y_scatter,
        mode="markers+text",
        marker=dict(
            color=cor_pontos,
            size=9,
            opacity=0.92,
            line=dict(color="white", width=1.2),
            symbol="circle",
        ),
        text=[_fmt_int(v) for v in valores_ano],
        textposition="middle right",
        textfont=dict(size=9, color=cor_pontos),
        hovertext=tooltips_pontos,
        hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
        yaxis=yaxis_scatter,
        xaxis="x",
    ))

    # Layout (padrão do gráfico de mortalidade)
    yaxis_cfg = dict(
        title=yaxis_title,
        title_font=dict(size=12),
        tickfont=dict(size=11),
        showgrid=False,
        showline=True,
        linecolor="#ccc",
        domain=[0, 1],
        type="category",
        categoryorder="array",
        categoryarray=categorias,
        showticklabels=True,
        automargin=True,
    )
    if force_y_ticklabels:
        yaxis_cfg.update(
            tickmode="array",
            tickvals=categorias,
            ticktext=categorias,
        )

    layout_kwargs = dict(
        title=dict(
            text=titulo,
            font=dict(size=16, family="Arial", color="#1a1a2e"),
            x=0.5, xanchor="center",
            pad=dict(b=80),
        ),
        height=height,
        width=width,
        plot_bgcolor="#f8f9fb",
        paper_bgcolor="white",
        margin=dict(l=110, r=180 if usar_y2 else 60, t=80, b=80),
        hovermode="closest",
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.0,
            xanchor="center", x=0.5,
            font=dict(size=11),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ddd", borderwidth=1,
        ),
        xaxis=dict(
            title=xaxis_title,
            tickvals=xticks_vals,
            ticktext=[_fmt_int(v) for v in xticks_vals],
            tickfont=dict(size=10),
            title_font=dict(size=12),
            gridcolor="#dde3ed",
            gridwidth=1,
            showline=True,
            linecolor="#ccc",
            zeroline=True,
            zerolinecolor="#bbb",
            range=[0, x_max_data * 1.25],
        ),
        yaxis=yaxis_cfg,
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#aaa",
            font=dict(size=12, family="Arial"),
        ),
    )
    if usar_y2:
        layout_kwargs["yaxis2"] = dict(
            title=yaxis2_title,
            title_font=dict(size=12),
            tickfont=dict(size=10),
            overlaying="y",
            side="right",
            tickvals=list(range(ano_ini, ano_fim + 1, int(anos_passo))),
            showgrid=False,
            showline=True,
            linecolor="#ccc",
            range=[ano_ini - 0.5, ano_fim + 0.5],
        )

    fig.update_layout(**layout_kwargs)

    # Linhas horizontais suaves (alinhadas às categorias)
    fig.update_layout(
        shapes=[
            dict(
                type="line",
                x0=0, x1=1, xref="paper",
                y0=cat, y1=cat, yref="y",
                line=dict(color="#dde3ed", width=1, dash="dot"),
            )
            for cat in categorias
        ]
    )

    fig.show()
    return fig



# Plot: Mortalidade

plot_duplo_plotly(
    por_categoria=por_faixa_mort.reindex(FAIXAS_ETARIAS_MORTALIDADE),
    por_ano=por_ano_mort_s,
    titulo=TITULO_MORTALIDADE,
    xaxis_title="Número de mortes",
    yaxis_title="Faixa etária",
    yaxis2_title="Ano",
    anos_range=ANOS_RANGE_MORTALIDADE,
    anos_passo=ANOS_PASSO_MORTALIDADE,
    usar_y2=True,
 )
print("✅ Mortalidade plotada")


# Plot: Casos (4 variáveis)

for g in graficos_casos:
    if g["titulo"] == "Casos - Ano Diagnóstico":
        plot_ano_diagnostico(
            por_ano=g["por_categoria"],
            titulo=f"{g['titulo']} (total por ano)",
        )
    else:
        plot_duplo_plotly(
            por_categoria=g["por_categoria"],
            por_ano=g["por_ano"],
            titulo=f"{g['titulo']}",
            xaxis_title="Número de casos",
            yaxis_title=g["ylabel_categoria"],
            yaxis2_title="Ano",
            anos_range=g["anos_range"],
            anos_passo=g["anos_passo"],
            usar_y2=True,
        )